# 03 — Synthesis Prompt Iteration

Exercises `synthesis.py`'s grounded-answer prompt across all three required
demo scenarios.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

import llm_client

if not llm_client.is_available():
    print(f"⚠️  Ollama / {llm_client.DEFAULT_MODEL} not reachable — run `ollama serve` first.")
else:
    print("✅ Ollama reachable.")

## Load the agent against a scratch DB (won't touch your real `forms.sqlite`)

In [ ]:
from agent import FormAgent

SAMPLE_DIR = os.path.abspath("../data/sample_forms")

agent = FormAgent(db_path=":memory:", chroma_dir="../chroma_db_notebook_scratch")
report = agent.load_directory(SAMPLE_DIR)
print(f"Loaded {len(report.loaded)} form(s): {report.loaded}")
if report.failed:
    print(f"Failed {len(report.failed)} form(s): {report.failed}")

## Demo 1 — Single-form Q&A

Try a few phrasings of the same underlying question against one form.
Watch for: does the answer stay grounded in that form's actual content,
and does `cited_form_ids` correctly point back to just that one form
(not accidentally citing others)?

In [ ]:
first_id = report.loaded[0] if report.loaded else None
print("Using form_id:", first_id)

single_form_questions = [
    "What is the status of this application?",
    "Was this application approved or rejected, and why?",
    "Summarize the remarks field in one sentence.",
]

for q in single_form_questions:
    result = agent.answer_question(q, form_id=first_id)
    print(f"Q: {q}")
    print(f"A: {result.answer}")
    print(f"   cited: {result.cited_form_ids}")
    print()

## Demo 2 — Summary

Run the summary path for every loaded form. Watch for: does the summary
include every structured field that matters (status/diagnosis), or does
it over-focus on the free-text field and drop structured facts?

In [ ]:
for form_id in report.loaded:
    result = agent.summarize_form(form_id)
    print(f"--- {form_id} ---")
    print(result.answer)
    print()

## Demo 3 — Multi-form holistic

Open-ended questions across the whole corpus. Watch for: does the answer
cite multiple `form_id`s when the question genuinely spans several forms,
and does it say "I don't have enough information" rather than guessing
when the corpus doesn't actually support a holistic claim?

In [ ]:
multi_form_questions = [
    "What patterns show up across all the forms?",
    "How many applications were approved versus rejected?",
    "Are there any common themes in the hospital admission notes?",
]

for q in multi_form_questions:
    result = agent.multi_form_query(q)
    print(f"Q: {q}")
    print(f"A: {result.answer}")
    print(f"   cited: {result.cited_form_ids}")
    print()

## Fast-iterate on `synthesis.py` without restarting the kernel

Edit the prompt template / citation-parsing logic in `src/synthesis.py`,
save the file, then run this cell and re-run any of the demo cells above
to see the effect immediately.

In [ ]:
import importlib
import synthesis
import retrieval
importlib.reload(synthesis)
importlib.reload(retrieval)
print("Reloaded synthesis.py and retrieval.py — re-run the demo cells above to test your change.")